---
title: "Stop Data Processing"
authors:
    - name: "Urwa Irfan"
# exports:
#     - format: pdf
#     - template: arxiv_one_column
#       output: exports/paper.pdf
---

In [1]:
# to run this notebook from terminal
# jupyter nbconvert --execute --to notebook --inplace scripts/stop_data_processing.ipynb

In [2]:
# load packages and utility functions
suppressMessages(library(tidytable))
suppressMessages(library(tidyverse))
source("utils.R")

In [3]:
# set constants
start_date = ymd("2021-03-01")
end_date = ymd("2026-04-30")

In [4]:
# load raw data
df_new = suppressMessages(read_csv("../data/raw/stop_data_2024_current.csv"))
df_old = suppressMessages(read_csv("../data/raw/stop_data_2020_2023.csv"))

Warning message:
"One or more parsing issues, call `problems()` on your data frame for details, e.g.:
  dat <- vroom(...)
  problems(dat)"


Warning message:
"One or more parsing issues, call `problems()` on your data frame for details, e.g.:
  dat <- vroom(...)
  problems(dat)"


In [5]:
# actions taken are split into force vs non-force in the newer dataset, but not the older
# this code sorts the actions taken in the older dataset into two columns to match the newer dataset

new_nonforce = unique_stop_data_column(df_new, "NonForceActionsTaken")
new_force = unique_stop_data_column(df_new, "ForceActionsTaken")
old = unique_stop_data_column(df_old, "ActionsTaken")
old_unique = setdiff(old, c(new_nonforce, new_force)) # unique actions not in the newer dataset

# manually analyzing old_unique, only the item containing "search" is a non-force action
nonforce_actions = c(new_nonforce, old_unique[str_detect(old_unique, "search")])
force_actions = c(new_force, old_unique[(!str_detect(old_unique, "search"))])

df_old_actions = df_old |>
    separate_longer_delim(ActionsTaken, delim = "\\|") |>
    mutate(ActionsTaken = str_trim(ActionsTaken)) |> # strip whitespaces
    mutate(
        type = case_when(
            ActionsTaken %in% force_actions ~ "force",
            ActionsTaken %in% nonforce_actions ~ "nonforce",
        )
    ) |>
    group_by(LEA_RecordID) |>
    summarise(
        ForceActionsTaken = str_c(ActionsTaken[type == "force"], collapse = "|"),
        NonForceActionsTaken = str_c(ActionsTaken[type == "nonforce"], collapse = "|")
    ) |>
    mutate(
        NonForceActionsTaken = ifelse(NonForceActionsTaken %in% c("", NA), "None", NonForceActionsTaken),
        ForceActionsTaken = ifelse(ForceActionsTaken %in% c("", NA), "None", ForceActionsTaken),
        ) |>
    select(LEA_RecordID, NonForceActionsTaken, ForceActionsTaken)


In [6]:
# process old dataset (2020-2023)
df_old_cleaned = df_old |>
  mutate(
    DateTimeOfStop = mdy_hm(DateTimeOfStop, tz = "America/Los_Angeles"),
    RespToServiceCall = ifelse(RespToServiceCall == "Yes", "Y", "N"),
    TypeOfStop = ifelse(
      TypeOfStop == "Vehicle", "Vehicular Stop", str_c(TypeOfStop, " Stop")
    ),
    LocationType = "LatLonPt",
  ) |>
  filter(
    # old data ends around mid-2024, exclude the 2024 data
    DateTimeOfStop >= start_date, year(DateTimeOfStop) <= 2023,
    City == "Berkeley",
  ) |>
  left_join(df_old_actions, by = "LEA_RecordID") |>
  select(LEA_Record_ID = LEA_RecordID, PersonNum, DateTimeOfStop,
    CallForService = RespToServiceCall, TypeOfStopDesc = TypeOfStop, StopDuration = DurationOfStop,
    City, LocationType, Latitude, Longitude, LocationDescription,
    TrafficViolationType, NonForceActionsTaken, ForceActionsTaken, PropertySeized, WeaponsFound, ContrabandOrEvidence, ResultOfStop,
    WarningOffense, CitationForInfractionOffense = CitationForInfraction,
    InfieldCiteRelOffense = InFieldCiteAndRelease, CustodialArrestNoWarrantOffense = CustodialArrestNoWarrant,
  )

# df_old_cleaned |>
#   summarise(min_date = min(DateTimeOfStop), max_date = max(DateTimeOfStop))
#   head(n = 5)

In [7]:
# process new dataset (2024-current)
df_new_cleaned = df_new |>
  mutate(
    DateTimeOfStop = mdy_hms(str_c(StopDate, " ", StopTime), tz = "America/Los_Angeles"),
    City = str_to_title(City),
    WarningOffense = case_when(
      is.na(VerbalWarningOffense) & is.na(WrittenWarningOffense) ~ NA,
      is.na(VerbalWarningOffense) & !(is.na(WrittenWarningOffense)) ~ WrittenWarningOffense,
      !(is.na(VerbalWarningOffense)) & is.na(WrittenWarningOffense) ~ VerbalWarningOffense,
      !(is.na(VerbalWarningOffense)) & !(is.na(WrittenWarningOffense)) ~ str_c(VerbalWarningOffense, "|", WrittenWarningOffense),
    )
  ) |>
  filter(
    DateTimeOfStop <= end_date, # new data starts at 2024
    City == "Berkeley"
  ) |> 
  select(LEA_Record_ID, PersonNum, DateTimeOfStop,
    CallForService, TypeOfStopDesc, StopDuration,
    City, LocationType, Latitude, Longitude,
    BlockNumber, StreetName, CrossStreet1, CrossStreet2, Highway, ClosestExit, OtherLocationDesc, # to be updated once i know how location data is processed
    TrafficViolationType, NonForceActionsTaken, ForceActionsTaken, PropertySeized, WeaponsFound, ContrabandOrEvidence, ResultOfStop,
    WarningOffense, CitationForInfractionOffense, InfieldCiteRelOffense, CustodialArrestNoWarrantOffense
  )

# df_new_cleaned |>
#   select(WarningOffense)
#   summarise(min_date = min(DateTimeOfStop), max_date = max(DateTimeOfStop))


In [ ]:
# join the two datasets
df = full_join(df_old_cleaned, df_new_cleaned) |>
    # modify date format for output
    mutate(DateTimeOfStop = format(DateTimeOfStop, "%Y-%m-%d %H:%M:%S%z"))

write_csv(df, "../data/clean/stop_data.csv")

Joining with `by = join_by(LEA_Record_ID, PersonNum, DateTimeOfStop, CallForService, TypeOfStopDesc, StopDuration,
City, LocationType, Latitude, Longitude, TrafficViolationType, NonForceActionsTaken, ForceActionsTaken,
PropertySeized, WeaponsFound, ContrabandOrEvidence, ResultOfStop, WarningOffense, CitationForInfractionOffense,
InfieldCiteRelOffense, CustodialArrestNoWarrantOffense)`
